# Annotated Transformer — Colab Training

Train the Multi30k German–English transformer in Google Colab.

**Before running:**
1. Go to **Runtime → Change runtime type → Hardware accelerator → GPU** (recommended).
2. Run cells top to bottom.

In [ ]:
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Clone repository

In [ ]:
REPO_URL = "https://github.com/vrajpatel04/Annotated-Transformer.git"
REPO_DIR = "Annotated-Transformer"

!rm -rf {REPO_DIR}
!git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 2. Install dependencies

Colab already includes PyTorch with CUDA. Remaining packages and spaCy language models are installed below.

In [ ]:
!uv pip install -q spacy sacrebleu plotly pyyaml GPUtil pandas datasets

!uv pip install -q https://github.com/explosion/spacy-models/releases/download/de_core_news_sm-3.8.0/de_core_news_sm-3.8.0-py3-none-any.whl
!uv pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

import sys
from pathlib import Path

repo_src = Path.cwd() / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

print("Dependencies ready.")

## 3. Train

Adjust the flags below as needed. Outputs are written to `outputs/`.

In [ ]:
EPOCHS = 8
BATCH_SIZE = 32
DEVICE = "cuda"  # use "cpu" if no GPU

!PYTHONPATH=src python -m transformer.cli \
    --mode train \
    --config configs/default.yaml \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --device {DEVICE} \
    --output-dir outputs

## 4. View results (optional)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import json

output_dir = Path("outputs")

metrics_path = output_dir / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    epochs = metrics.get("epochs", [])
    if epochs:
        print("Final epoch metrics:")
        for key, value in epochs[-1].items():
            print(f"  {key}: {value}")
    else:
        print("No epoch metrics found in metrics.json")

dashboard_path = output_dir / "dashboard.html"
if dashboard_path.exists():
    display(HTML(dashboard_path.read_text()))

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

output_dir = Path("outputs")
if output_dir.exists():
    archive = shutil.make_archive("transformer_outputs", "zip", output_dir)
    files.download(archive)
    print("Downloaded transformer_outputs.zip")
else:
    print("No outputs/ directory found — run training first.")